In [1]:
import os
import pandas as pd
import geopandas as gpd

In [2]:
DATA_DIR = os.path.join("..", "data", "ethiopia")

In [3]:
global_conflict_fp = os.path.join(DATA_DIR, "GEDEvent_v25_1.csv")

# read CSV and create GeoDataFrame from latitude/longitude columns
df = pd.read_csv(global_conflict_fp, low_memory=False)
global_conflict_gdf = gpd.GeoDataFrame(
    df, 
    geometry=gpd.points_from_xy(df.longitude, df.latitude),
    crs='EPSG:4326'
)

In [4]:
# Load Ethiopia boundary (using woredas to get the overall extent)
ethiopia_boundary_fp = os.path.join(DATA_DIR, "woredas.json")
woreda_gdf = gpd.read_file(ethiopia_boundary_fp)

# Filter conflict data to Ethiopia (2022-2024)
# First, filter by country
ethiopia_conflict_gdf = global_conflict_gdf[global_conflict_gdf['country'] == 'Ethiopia'].copy()

# Then filter by year (2022-2024)
ethiopia_conflict_gdf = ethiopia_conflict_gdf[
    (ethiopia_conflict_gdf['year'] >= 2020) & 
    (ethiopia_conflict_gdf['year'] <= 2024)
]

print(f"Total global conflicts: {len(global_conflict_gdf)}")
print(f"Ethiopia conflicts (2022-2024): {len(ethiopia_conflict_gdf)}")

# Spatial join to add woreda information
# Select and rename the columns we need from woreda_gdf
woreda_cols = woreda_gdf[['GID_3', 'NAME_1', 'NAME_2', 'NAME_3', 'geometry']].copy()
woreda_cols = woreda_cols.rename(columns={
    'GID_3': 'code',
    'NAME_1': 'region',
    'NAME_2': 'zone',
    'NAME_3': 'woreda'
})

# Perform spatial join (conflict points within woreda polygons)
ethiopia_conflict_gdf = ethiopia_conflict_gdf.sjoin(
    woreda_cols, 
    how='left', 
    predicate='within'
).drop(columns=['index_right'])

print(f"Conflicts with woreda info: {ethiopia_conflict_gdf['code'].notna().sum()}")

Total global conflicts: 385918
Ethiopia conflicts (2022-2024): 3317
Conflicts with woreda info: 3315


In [5]:
ethiopia_conflict_gdf["woreda"].value_counts()

woreda
Liben           71
Kobo            62
KaftaHumera     52
SahartiSamre    51
Dera            47
                ..
DaroLebu         1
Seru             1
Agarfa           1
Amibara          1
Dodola           1
Name: count, Length: 375, dtype: int64

In [6]:
ethiopia_conflict_gdf["zone"].value_counts()

zone
NorthShewa      259
MirabShewa      209
Mehakelegnaw    182
Debubawi        180
Guji            172
               ... 
Liben             1
DireDawa          1
DebubOmo          1
Hadiya            1
Gedeo             1
Name: count, Length: 61, dtype: int64

In [7]:
# Save to CSV
output_fp = os.path.join(DATA_DIR, "ethiopia_conflict_data.csv")
ethiopia_conflict_gdf.to_csv(output_fp, index=False)

print(f"Saved {len(ethiopia_conflict_gdf)} conflicts to {output_fp}")

Saved 3317 conflicts to ..\data\ethiopia\ethiopia_conflict_data.csv
